In [1]:
import pandas as pd
import tabula
import pdfplumber
import re

import os
print(f"Jupyter sedang di: {os.getcwd()}")

Jupyter sedang di: c:\Users\hpamd\OneDrive\Desktop\proyek-batang-data\notebooks


In [2]:
pdf_path = "../data/raw/BPS3325-indeks_pembangunan_gender_kabupaten_batang_2025.pdf"

try:
    tables = tabula.read_pdf(pdf_path, pages='all', multiple_tables=True)

    print(f"Berhasil menemukan {len(tables)} tabel dalam PDF ini.")

    if len(tables) > 0:
        display(tables[0].head())
    else:
        print("Tidak ada tabel yang terdeteksi. Mungkin ini halaman teks.")

except Exception as e:
    print(f"Waduh, ada error: {e}")


Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'
Got stderr: Nov 06, 2026 3:39:44 PM org.apache.fontbox.ttf.CmapSubtable processSubtype14
Nov 06, 2026 3:39:44 PM org.apache.fontbox.ttf.CmapSubtable processSubtype14



Berhasil menemukan 71 tabel dalam PDF ini.


,Unnamed: 0


In [3]:
for i in range(10):
    print(f"\n{'='*20} TABEL KE-{i} {'='*20}")
    display(tables[i].head(3))


==================== TABEL KE-0 ====================


,Unnamed: 0



==================== TABEL KE-1 ====================


,v,v.1



==================== TABEL KE-2 ====================


,vi,vi.1


... (10 outputs left out)


==================== TABEL KE-8 ====================


,8,8.1



==================== TABEL KE-9 ====================


,9,9.1


In [4]:
# Cari di tabel nomor berapa kata kunci "Gender" atau "IPG" berada
kata_kunci = ["Gender", "IPG", "Indeks Pembangunan"]

for i, table in enumerate(tables):
    # Ubah seluruh isi tabel jadi teks (string) untuk dicari
    table_str = table.to_string().lower()

    # Cek apakah salah satu kata kunci ada di tabel ini
    if any(kata.lower() in table_str for kata in kata_kunci):
        print(f"Ditemukan di TABEL ke-{i}")

        display(table.head(5))
        print("-"*50)

Ditemukan di TABEL ke-4


,Gambar 3.1,"Piramida Penduduk Kabupaten Batang, 2025",Unnamed: 0,15
0,Gambar 3.2,Tingkat Partisipasi Angkatan Kerja (TPAK) menurut,NaN,22.0
1,NaN,Jenis Kelamin di Kabupaten Batang Tahun 2021-2025,NaN,NaN
2,NaN,NaN,NaN,NaN
3,Gambar 3.3,Tingkat Pengangguran Terbuka (TPT) menurut Jenis,NaN,24.0
4,NaN,Kelamin di Kabupaten Batang Tahun 2021-2025,NaN,NaN


--------------------------------------------------
Ditemukan di TABEL ke-37


,Unnamed: 0,Unnamed: 1,Unnamed: 2,IPM,IPM.1
0,Kabupaten/Kota IPG,NaN,IPM,NaN,NaN
1,NaN,NaN,NaN,Laki-Laki,Perempuan
2,(1) (2),NaN,(3),(4),(5)
3,"Cilacap 88,22",NaN,"73,45","78,25","69,03"
4,"Banyumas 90,01",NaN,"75,44","79,34","71,41"


--------------------------------------------------
Ditemukan di TABEL ke-39


,Kabupaten/Kota,IPG,IPM,Unnamed: 0,IPM.1,Unnamed: 1,Unnamed: 2,IPM.2,Unnamed: 3,Unnamed: 4
0,Kabupaten/Kota,IPG,IPM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,Laki-Laki,Perempuan,NaN,NaN
2,(1),(2),(3),(4),(5),NaN,NaN,NaN,NaN,NaN
3,Tegal,Tegal,"89,17","89,17","72,36","72,36","77,31","77,31","68,94","68,94"
4,Brebes,"87,97","71,18","75,62","66,52",NaN,NaN,NaN,NaN,NaN


--------------------------------------------------
Ditemukan di TABEL ke-40


,IPG
0,IPG


--------------------------------------------------


In [5]:
# Ekstrak tabel 39
# Perkiraan tabel 39 ada di halaman sekitar 47-50

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()

        # Cari halaman yang ada Tegal, Brebes, dan IPG
        if text and "Tegal" in text and "Brebes" in text and "IPG" in text:
            print(f"Data ditemukan di Halaman PDF ke-{i+1} (Index: {i})")

            tables = page.extract_tables()

            if tables:
                df_rapi = pd.DataFrame(tables[0])
                display(df_rapi.head(10))
                break

Data ditemukan di Halaman PDF ke-63 (Index: 62)


,0,1,2,3,4,5,6,7,8
0,Kabupaten/Kota,IPG,IPM,,IPM,,,IPM,
1,NaN,NaN,NaN,NaN,Laki-Laki,NaN,NaN,Perempuan,NaN
2,(1),(2),(3),(4),NaN,NaN,(5),NaN,NaN
3,Tegal,"89,17","72,36","77,31",NaN,NaN,"68,94",NaN,NaN
4,Brebes,"87,97","71,18","75,62",NaN,NaN,"66,52",NaN,NaN
5,Kota Magelang,"97,09","82,92","84,53",NaN,NaN,"82,07",NaN,NaN
6,Kota Surakarta,"97,52","84,92","85,76",NaN,NaN,"83,63",NaN,NaN
7,Kota Salatiga,"95,70","86,23","89,16",NaN,NaN,"85,33",NaN,NaN
8,Kota Semarang,"96,38","85,80","87,57",NaN,NaN,"84,40",NaN,NaN
9,Kota Pekalongan,"95,65","77,76","80,52",NaN,NaN,"77,02",NaN,NaN


In [8]:
# Kita tentukan dua halaman: Index 61 (Halaman 62) dan 62 (Halaman 63)
halaman_target = [61, 62]
data_rapi = []

with pdfplumber.open(pdf_path) as pdf:
    for page_idx in halaman_target:
        page = pdf.pages[page_idx]
        text = page.extract_text()

        if text:
            # Pecah teks per baris
            lines = text.split("\n")

            for line in lines:
                # Cari baris yang ada pola angka desimal (misal: 89,17)
                if re.search(r"\d+,\d+", line):
                    # Pecah baris berdasarkan spasi
                    parts = line.split()

                    # Cari di mana angka pertamanya dimulai
                    try:
                        idx_angka = next(
                            i
                            for i, part in enumerate(parts)
                            if re.match(r"\d+,\d+", part)
                        )
                        nama_daerah = " ".join(parts[:idx_angka])
                        angka = parts[idx_angka:]

                        # Pastikan jumlah angkanya pas (ada 4 angka: IPG, IPM, IPM L, IPM P)
                        if len(angka) >= 4:
                            # Bersihkan nama daerah dari karakter aneh (jika ada)
                            nama_daerah = nama_daerah.replace("|", "").strip()

                            # Masukkan ke list
                            data_rapi.append([nama_daerah] + angka[:4])
                    except StopIteration:
                        # Abaikan baris yang tidak punya pola angka yang jelas
                        pass

# Masukkan ke DataFrame
df_final = pd.DataFrame(
    data_rapi,
    columns=["Kabupaten/Kota", "IPG", "IPM", "IPM Laki-laki", "IPM Perempuan"],
)

# Masukkan ke DataFrame
df_final = pd.DataFrame(
    data_rapi,
    columns=["Kabupaten/Kota", "IPG", "IPM", "IPM Laki-laki", "IPM Perempuan"],
)

# --- PROSES CLEANING AKHIR (VERSI ANTI-ERROR) ---

# 1. Ubah koma jadi titik KHUSUS untuk kolom angka
for col in ["IPG", "IPM", "IPM Laki-laki", "IPM Perempuan"]:
    df_final[col] = df_final[col].astype(str).str.replace(",", ".")

# 2. Ubah tipe data jadi float, dan jika ada yang gagal (errors='coerce'), ubah jadi NaN (kosong)
for col in ["IPG", "IPM", "IPM Laki-laki", "IPM Perempuan"]:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

# 3. Buang baris yang IPG-nya NaN (ini akan membuang judul tabel, baris kosong, dll)
df_final = df_final.dropna(subset=["IPG"])

# 4. Hapus baris duplikat (kalau ada yang kebaca dua kali)
df_final = df_final.drop_duplicates()

# 5. Reset index agar urut dari 0 lagi
df_final = df_final.reset_index(drop=True)

print("=== DATA GABUNGAN BERHASIL DIBERSIHKAN ===")
display(df_final)

# Simpan ke folder data/processed/
output_path = "../data/processed/ipg_jateng_2025.csv"
df_final.to_csv(output_path, index=False)

print(f"Mantap! Data berhasil disimpan di: {output_path}")

=== DATA GABUNGAN BERHASIL DIBERSIHKAN ===


,Kabupaten/Kota,IPG,IPM,IPM Laki-laki,IPM Perempuan
0,Cilacap,88.22,73.45,78.25,69.03
1,Banyumas,90.01,75.44,79.34,71.41
2,Purbalingga,94.34,71.71,75.11,70.86
3,Banjarnegara,95.15,70.61,72.54,69.02
4,Kebumen,93.60,73.17,76.62,71.72
5,Purworejo,95.58,75.93,78.23,74.77
6,Wonosobo,93.22,71.50,NaN,69.86
7,Magelang,93.31,73.09,NaN,72.00
8,Boyolali,94.90,76.95,80.51,76.40
9,Klaten,96.35,78.68,NaN,80.65


Mantap! Data berhasil disimpan di: ../data/processed/ipg_jateng_2025.csv
